# 04. SQL Integration (DuckDB)

**Purpose:** Rebuild the disposable income model in SQL (CTEs, joins, window functions) and cross-check against the Python model.

**Inputs:** `data/clean/*.csv`  |  **Query:** `sql/disposable_income.sql`

## load tables into DuckDB

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()
tables = ["fair_work_rates", "rent", "tax_rates", "lito", "insurance",
          "transport", "food_scenarios", "living_costs_other"]
for t in tables:
    con.execute(f"CREATE TABLE {t} AS SELECT * FROM read_csv_auto('../data/clean/{t}.csv')")

con.execute("SHOW TABLES").df()

,name
0,fair_work_rates
1,food_scenarios
2,insurance
3,lito
4,living_costs_other
5,rent
6,tax_rates
7,transport


## run the SQL file

In [2]:
with open("../sql/disposable_income.sql", encoding="utf-8") as f:
    query = f.read()

sql_result = con.execute(query).df()
sql_result

,city,industry,disposable_share,disposable_alone,rank_share
0,Adelaide,Hospitality,200.21,-14.98,1
1,Melbourne,Hospitality,195.11,-2.32,2
2,Brisbane,Hospitality,169.61,-40.76,3
3,Perth,Hospitality,168.21,-65.43,4
4,Sydney,Hospitality,29.61,-180.98,5
5,Adelaide,Retail,259.76,44.57,1
6,Melbourne,Retail,254.66,57.23,2
7,Brisbane,Retail,229.16,18.79,3
8,Perth,Retail,227.76,-5.88,4
9,Sydney,Retail,89.16,-121.43,5


## cross-check against Python model

In [4]:
model = pd.read_csv("../data/clean/model_base.csv")
py = model.query("residency == 'Resident' and rent_shock == 0")

for housing, col in [("Share room", "disposable_share"), ("Living alone", "disposable_alone")]:
    a = sql_result.set_index(["city", "industry"])[col]
    b = py[py["housing"] == housing].set_index(["city", "industry"])["disposable_weekly"]
    diff = (a - b.reindex(a.index)).abs().max()
    print(f"{housing}: max difference SQL vs Python = {diff:.2f}")
    assert diff < 0.02

print("SQL and Python models agree")

Share room: max difference SQL vs Python = 0.00
Living alone: max difference SQL vs Python = 0.00
SQL and Python models agree


## Techniques
CTEs for each model step, joins across eight tables, window functions (`ROW_NUMBER` to pick the tax bracket and offset band, `RANK` for city ranking). The SQL output matches the Python model within two cents.